In [6]:
import torch
import torch.nn as nn
import math
import numpy as np

In [7]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding.
    Adds temporal position information to frame features.
    x = x + positional_encoding
    """
    def __init__(self, d_model: int = 512, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)   # even dims
        pe[:, 1::2] = torch.cos(position * div_term)   # odd dims

        self.register_buffer('pe', pe.unsqueeze(0))    # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, d_model)
        Returns:
            x: (B, T, d_model) with positional encoding added
        """
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

In [8]:
class InputProjection(nn.Module):
    """
    Projects CNN features (1024-dim) down to d_model (512-dim).
    Required because input_dim (1024) != d_model (512).

    Without this, the Transformer would crash — it expects
    vectors of exactly d_model dimensions as input.

    Architecture:
        Linear(1024 → 512)
        LayerNorm(512)
        ReLU
    """
    def __init__(self, input_dim: int = 1024, d_model: int = 512):
        super().__init__()
        self.proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.ReLU()
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, 1024) — raw CNN features
        Returns:
            (B, T, 512)     — projected features
        """
        return self.proj(x)

In [9]:
class TemporalTransformerEncoder(nn.Module):
    """
    Temporal Transformer Encoder for video representation learning.
    Optimized for 149 videos dataset.

    Flow:
        CNN features (B, T, 1024)
            → InputProjection: Linear(1024→512) + LayerNorm + ReLU
            → PositionalEncoding: x = x + positional_encoding
            → 3x TransformerEncoderLayer (multi-head self-attention + FFN)
            → contextual_features (B, T, 512)

    Config (chosen for 149 videos):
        input_dim   = 1024   (GoogleNet output)
        d_model     = 512    (transformer hidden dim)
        nhead       = 8      (512/8 = 64 dim per head)
        num_layers  = 3      (same depth as TR-SUM)
        dim_feedfwd = 2048   (4 × d_model)
        dropout     = 0.1

    Args:
        input_dim   : CNN feature dimension (default: 1024)
        d_model     : transformer hidden dimension (default: 512)
        nhead       : number of attention heads (default: 8)
        num_layers  : number of stacked encoder layers (default: 3)
        dim_feedfwd : FFN hidden size (default: 2048 = 4 × d_model)
        dropout     : dropout rate (default: 0.1)
        max_len     : maximum sequence length (default: 5000)
        freeze      : if True, freeze all params for RL integration
    """
    def __init__(
        self,
        input_dim: int   = 1024,
        d_model: int     = 512,
        nhead: int       = 8,
        num_layers: int  = 3,
        dim_feedfwd: int = 2048,
        dropout: float   = 0.1,
        max_len: int     = 5000,
        freeze: bool     = False,
    ):
        super().__init__()
        assert d_model % nhead == 0, \
            f"d_model ({d_model}) must be divisible by nhead ({nhead})"

        self.d_model    = d_model
        self.input_dim  = input_dim
        self.num_layers = num_layers

        # 1. Project 1024 → 512 (required since input_dim != d_model)
        self.input_proj = InputProjection(
            input_dim = input_dim,
            d_model   = d_model,
        )

        # 2. Add temporal positional encoding: x = x + positional_encoding
        self.pos_enc = PositionalEncoding(
            d_model = d_model,
            max_len = max_len,
            dropout = dropout,
        )

        # 3. Stack of 3 Transformer encoder layers
        encoder_layer = nn.TransformerEncoderLayer(
            d_model        = d_model,
            nhead          = nhead,
            dim_feedforward= dim_feedfwd,
            dropout        = dropout,
            batch_first    = True,    # (B, T, d_model)
            norm_first     = True,    # pre-norm: more stable training
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers = num_layers,
            norm       = nn.LayerNorm(d_model),
        )

        # 4. Optional freeze for RL integration (Person 3)
        if freeze:
            self.freeze()

    def freeze(self):
        """Freeze all parameters — call before connecting to RL agent."""
        for p in self.parameters():
            p.requires_grad = False
        print("[TemporalEncoder] All parameters frozen.")

    def unfreeze(self):
        """Unfreeze all parameters — call to resume end-to-end training."""
        for p in self.parameters():
            p.requires_grad = True
        print("[TemporalEncoder] All parameters unfrozen.")

    def encode(
        self,
        features: torch.Tensor,
        padding_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        """
        Main encoding interface — called by Person 2/3's RL environment.

        Args:
            features     : (B, T, 1024) — precomputed CNN features
            padding_mask : (B, T) bool, True = padded position to ignore

        Returns:
            contextual_features: (B, T, 512)
        """
        # Step 1: Project 1024 → 512
        x = self.input_proj(features)                  # (B, T, 512)

        # Step 2: x = x + positional_encoding
        x = self.pos_enc(x)                            # (B, T, 512)

        # Step 3: Self-attention across all frames
        contextual_features = self.transformer(
            x,
            src_key_padding_mask = padding_mask,       # (B, T) or None
        )                                              # (B, T, 512)

        return contextual_features

    def forward(self, features, padding_mask=None):
        return self.encode(features, padding_mask)

In [10]:
def pad_and_mask(feature_list: list, d_input: int = 1024):
    """
    Pads a batch of variable-length videos to the same T
    and builds a padding mask.

    Args:
        feature_list : list of tensors, each (T_i, 1024)
        d_input      : CNN feature dimension (1024)

    Returns:
        padded : (B, T_max, 1024)
        mask   : (B, T_max) — True = padded (ignore this position)
    """
    B     = len(feature_list)
    T_max = max(f.shape[0] for f in feature_list)

    padded = torch.zeros(B, T_max, d_input)
    mask   = torch.ones(B, T_max, dtype=torch.bool)   # all padded initially

    for i, f in enumerate(feature_list):
        T_i = f.shape[0]
        padded[i, :T_i, :] = f
        mask[i, :T_i]      = False                    # real frames → attend

    return padded, mask

In [ ]:
# ── Config for 149 videos ──────────────────────────────────
INPUT_DIM   = 1024   # GoogleNet penultimate layer
D_MODEL     = 512    # transformer hidden dim (needs projection from 1024)
N_HEADS     = 8      # 512 / 8 = 64 dim per head
NUM_LAYERS  = 3      # same depth as TR-SUM
DIM_FF      = 2048   # 4 × d_model
DROPOUT     = 0.1

# ── Build model ────────────────────────────────────────────
encoder = TemporalTransformerEncoder(
    input_dim   = INPUT_DIM,
    d_model     = D_MODEL,
    nhead       = N_HEADS,
    num_layers  = NUM_LAYERS,
    dim_feedfwd = DIM_FF,
    dropout     = DROPOUT,
    freeze      = False,
)

total_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)

print("=" * 45)
print("  Temporal Transformer Encoder")
print("=" * 45)
print(f"  input_dim   : {INPUT_DIM}  (CNN features)")
print(f"  d_model     : {D_MODEL}   (projection: {INPUT_DIM}→{D_MODEL})")
print(f"  nhead       : {N_HEADS}     (head dim = {D_MODEL // N_HEADS})")
print(f"  num_layers  : {NUM_LAYERS}")
print(f"  dim_feedfwd : {DIM_FF}")
print(f"  dropout     : {DROPOUT}")
print(f"  Parameters  : {total_params:,}")
print("=" * 45)

In [ ]:
# Simulate 3 videos with different frame counts
video_features = [
    torch.randn(120, 1024),    # video 1: 120 frames
    torch.randn(85,  1024),    # video 2:  85 frames
    torch.randn(200, 1024),    # video 3: 200 frames
]

# Pad and build mask
padded, mask = pad_and_mask(video_features, d_input=INPUT_DIM)
print(f"Padded input shape : {padded.shape}")    # (3, 200, 1024)
print(f"Padding mask shape : {mask.shape}")      # (3, 200)

# Encode
encoder.eval()
with torch.no_grad():
    contextual_features = encoder.encode(padded, padding_mask=mask)

print(f"contextual_features: {contextual_features.shape}")   # (3, 200, 512)
print("Batch test passed.")

In [ ]:
def encode_single_video(encoder, features: torch.Tensor) -> torch.Tensor:
    """
    Encode a single video without batching.

    Args:
        encoder  : TemporalTransformerEncoder
        features : (T, 1024) — CNN features for one video

    Returns:
        contextual_features: (T, 512)
    """
    assert features.ndim == 2 and features.shape[1] == 1024, \
        "Expected shape (T, 1024)"

    x = features.unsqueeze(0)               # (1, T, 1024)
    encoder.eval()
    with torch.no_grad():
        out = encoder.encode(x)             # (1, T, 512)
    return out.squeeze(0)                   # (T, 512)


# Test
single_video = torch.randn(90, 1024)
out = encode_single_video(encoder, single_video)
print(f"Single video output : {out.shape}")    # (90, 512)
print("Single video test passed.")